## 1. Installation

In [1]:
!pip install -q marker-pdf
from importlib.metadata import version
print("marker-pdf version :", version("marker-pdf"))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 2.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 11.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 31.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 56.2 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.9 MB/s eta 0:00:00:00:01
   ━━

## 2. Verifier le dataset Kaggle

Sur Kaggle, les fichiers d'un dataset ajoute au notebook sont montes en lecture seule sous `/kaggle/input/<nom-du-dataset>/`. On liste le contenu pour confirmer le chemin exact avant de continuer (utile si le PDF est dans un sous-dossier).

In [3]:
import os

# Etape 1 : lister tous les datasets disponibles sous /kaggle/input
print("Datasets disponibles dans /kaggle/input :")
for name in os.listdir("/kaggle/input"):
    print(" -", name)

Datasets disponibles dans /kaggle/input :
 - datasets


In [4]:
# Etape 2 : exploration recursive complete pour trouver tous les PDF,
# quelle que soit la profondeur des sous-dossiers
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower().endswith(".pdf"):
            print(os.path.join(root, f))

/kaggle/input/datasets/hasnaasaadi/my-data/loan_0001.pdf


## 3. Configuration des parametres de lecture

Marker se configure via un dictionnaire passe a un `ConfigParser`, qui genere ensuite les options du `PdfConverter` (modeles, OCR, format de sortie...).

Point important, different de Docling : Docling expose texte + tableaux + structure a partir d'un seul objet `doc`. Marker, lui, produit une sortie differente selon `output_format` :

- `"markdown"` -> texte Markdown pret a l'emploi (tableaux deja en Markdown, mais sans hierarchie explicite)
- `"json"` -> structure hierarchique complete (`block_type`, `children`, `polygon` = bbox), utile pour detecter la structure du document et retrouver les tableaux precisement

On cree donc deux converters (un par format), en partageant les memes modeles (`create_model_dict()`) pour ne pas les recharger deux fois.

In [5]:
from marker.config.parser import ConfigParser
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict

# Modeles charges une seule fois, reutilises par les deux converters
artifact_dict = create_model_dict()

def build_converter(output_format: str) -> PdfConverter:
    config = {
        "output_format": output_format,
        # --- OCR ---
        # force_ocr = False car les PDF sont des PDF texte natifs (non scannes).
        # Marker lit directement le texte deja encode dans le PDF -> plus rapide, plus fiable.
        # Mettre True uniquement si le PDF est un scan (image pure, sans texte selectionnable).
        "force_ocr": False,
        "paginate_output": True,  # ajoute des marqueurs de page, utile pour la structure
    }
    config_parser = ConfigParser(config)
    return PdfConverter(
        config=config_parser.generate_config_dict(),
        artifact_dict=artifact_dict,
        processor_list=config_parser.get_processors(),
        renderer=config_parser.get_renderer(),  # indispensable pour que output_format soit respecte
    )

converter_md = build_converter("markdown")
converter_json = build_converter("json")
print("Converters prets (markdown + json).")

Converters prets (markdown + json).


## 4. Lecture du PDF

Chemin adapte a Kaggle : `/kaggle/input/<dataset>/...` pour la lecture (lecture seule).

In [6]:
# Adapte le nom de fichier si besoin, d'apres la sortie de la cellule 2
DATASET_DIR = "/kaggle/input/datasets/hasnaasaadi/my-data"
PDF_PATH = os.path.join(DATASET_DIR, "loan_0001.pdf")

rendered_md = converter_md(PDF_PATH)
print("Sortie markdown -> type :", type(rendered_md).__name__)

Running OCR Error Detection: 100%|██████████| 1/1 [00:00<00:00,  2.71it/s]
Detecting bboxes: 0it [00:00, ?it/s]
Recognizing tables: 100%|██████████| 1/1 [00:07<00:00,  7.50s/it]
Detecting bboxes: 0it [00:00, ?it/s]

Sortie markdown -> type : MarkdownOutput


## 5. Export du contenu

Le Markdown est le plus lisible pour un humain ; on le recupere via `text_from_rendered`.

In [7]:
from marker.output import text_from_rendered

md_text, ext, images = text_from_rendered(rendered_md)
print(md_text)



{0}------------------------------------------------

# **Loan Application for a Commercial Real Estate Loan - Company Profile**

## **1. Applicant**

| Company Name                       | Söding GmbH & Co. KGaA UG                   |
|------------------------------------|---------------------------------------------|
| Legal Form                         | Limited Partnership (KG)                    |
| Date of Incorporation              | 01/12/2006                                  |
| Business Address                   | Falk-Textor-Weg 160, 86608 Gransee, Germany |
| Commercial Register Number / Court | HRB 977141 / Gransee Local Court            |
| VAT ID / Tax Number                | DE681242882                                 |

## **2. Financial Request / Property Description**

| Type of Property                    | Office and Commercial Building                 |
|-------------------------------------|------------------------------------------------|
| Property Name       

In [8]:
# Sauvegarde en Markdown dans /kaggle/working (seul dossier ecriture autorisee sur Kaggle)
OUTPUT_DIR = "/kaggle/working"
md_out_path = os.path.join(OUTPUT_DIR, "loan_0001_marker.md")

with open(md_out_path, "w", encoding="utf-8") as f:
    f.write(md_text)

print("Sauvegarde OK :", md_out_path)

Sauvegarde OK : /kaggle/working/loan_0001_marker.md


In [9]:
# Sortie JSON (structure hierarchique + positions), utile pour extraire des champs automatiquement
import json

rendered_json = converter_json(PDF_PATH)
data = rendered_json.model_dump()
print("Cles principales :", list(data.keys()))

json_out_path = os.path.join(OUTPUT_DIR, "loan_0001_marker.json")
with open(json_out_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
print("JSON sauvegarde :", json_out_path)

Running OCR Error Detection: 100%|██████████| 1/1 [00:00<00:00,  2.81it/s]
Detecting bboxes: 0it [00:00, ?it/s]
Recognizing tables: 100%|██████████| 1/1 [00:07<00:00,  7.85s/it]
Detecting bboxes: 0it [00:00, ?it/s]

Cles principales : ['children', 'block_type', 'metadata']
JSON sauvegarde : /kaggle/working/loan_0001_marker.json


## 6. Parcourir les elements avec leur structure (block_type)

Chaque bloc a un `block_type` (titre de section, texte, tableau...) et un `polygon` (coordonnees sur la page), organises en arbre via `children`. Utile pour relier une valeur extraite a sa position et a la section a laquelle elle appartient.

In [10]:
import re

def iter_blocks(block, level=0):
    yield block, level
    for child in (block.children or []):
        yield from iter_blocks(child, level + 1)

for root_block in rendered_json.children:
    for block, level in iter_blocks(root_block):
        text_preview = re.sub("<[^<]+?>", " ", block.html or "").strip()
        if text_preview:
            indent = "  " * level
            print(f"{indent}[{block.block_type}] id={block.id} | bbox {block.polygon}")
            print(f"{indent}   {text_preview[:90]}")

  [SectionHeader] id=/page/0/SectionHeader/0 | bbox [[57.023616790771484, 60.84765625], [501.5835876464844, 60.84765625], [501.5835876464844, 105.25], [57.023616790771484, 105.25]]
     Loan Application for a Commercial Real Estate   Loan - Company Profile
  [SectionHeader] id=/page/0/SectionHeader/1 | bbox [[57.023616790771484, 133.4249267578125], [125.13671875, 132.7958984375], [125.13671875, 145.4249267578125], [57.023616790771484, 146.7744140625]]
     1. Applicant
  [Table] id=/page/0/Table/2 | bbox [[50.7589453125, 152.83451171875], [542.9129296875, 151.18998046875], [542.9129296875, 286.25533203125], [50.7589453125, 287.89986328125]]
     Company Name  Söding GmbH & Co. KGaA UG    Legal Form  Limited Partnership (KG)    Date of
    [TableCell] id=/page/0/TableCell/138 | bbox [[53.8903303265992, 153.54472354107688], [254.7703926604821, 153.54472354107688], [254.7703926604821, 176.79039378578494], [53.8903303265992, 176.79039378578494]]
       Company Name
    [TableCell] id=/page

## 7. Recuperer les tableaux

Le formulaire de credit est surtout compose de tableaux (etiquette / valeur). Dans la sortie JSON, chaque tableau est un bloc `block_type == "Table"` contenant du HTML ; on le convertit directement en DataFrame pandas avec `pandas.read_html`.

In [11]:
import io
import pandas as pd

def find_tables(blocks):
    for block in blocks:
        if block.block_type == "Table":
            yield block
        yield from find_tables(block.children or [])

all_tables = list(find_tables(rendered_json.children))
print("Nombre de tableaux detectes :", len(all_tables))

for i, table_block in enumerate(all_tables):
    df = pd.read_html(io.StringIO(table_block.html))[0]
    print(f"\n--- Tableau {i+1} ---")
    print(df)

Nombre de tableaux detectes : 3

--- Tableau 1 ---
                         Company Name  \
0                          Legal Form   
1               Date of Incorporation   
2                    Business Address   
3  Commercial Register Number / Court   
4                 VAT ID / Tax Number   

                     Söding GmbH & Co. KGaA UG  
0                     Limited Partnership (KG)  
1                                   01/12/2006  
2  Falk-Textor-Weg 160, 86608 Gransee, Germany  
3             HRB 977141 / Gransee Local Court  
4                                  DE681242882  

--- Tableau 2 ---
                      Type of Property  \
0                        Property Name   
1                               Adress   
2  Purchase Price / Construction Costs   
3             Desired Financing Amount   
4                       Purpose of Use   
5                  Equity Contribution   
6                 Year of Construction   
7                           Total Area   

          

In [12]:
with open(md_out_path, "r", encoding="utf-8") as f:
    print(f.read())



{0}------------------------------------------------

# **Loan Application for a Commercial Real Estate Loan - Company Profile**

## **1. Applicant**

| Company Name                       | Söding GmbH & Co. KGaA UG                   |
|------------------------------------|---------------------------------------------|
| Legal Form                         | Limited Partnership (KG)                    |
| Date of Incorporation              | 01/12/2006                                  |
| Business Address                   | Falk-Textor-Weg 160, 86608 Gransee, Germany |
| Commercial Register Number / Court | HRB 977141 / Gransee Local Court            |
| VAT ID / Tax Number                | DE681242882                                 |

## **2. Financial Request / Property Description**

| Type of Property                    | Office and Commercial Building                 |
|-------------------------------------|------------------------------------------------|
| Property Name       

## Resume des chemins Kaggle

- Lecture (input, lecture seule) : `/kaggle/input/<dataset>/...`
- Ecriture (output) : `/kaggle/working/...` -> c'est ce dossier qui apparait ensuite dans l'onglet "Output" du notebook Kaggle une fois execute/commit

## Resume des parametres principaux

- `force_ocr = False` : PDF texte natif, pas de scan -> lecture directe, plus rapide (mettre `True` pour un scan)
- `output_format="markdown"` pour le texte pret a l'emploi, `output_format="json"` pour la structure hierarchique complete + les tableaux
- `renderer=config_parser.get_renderer()` est indispensable : sans lui, Marker ignore `output_format` et renvoie toujours du Markdown
- Sorties utiles : `text_from_rendered()`, `rendered_json.model_dump()`, `block.block_type`, `block.children`, `block.polygon`, `block.html`